## Importación de librerías necesarias

In [ ]:
import pandas as pd
import plotly.graph_objects as go 

### Análisis exploratorio de los datos

In [ ]:
data = pd.read_csv('DataScience_salaries_2025.csv')
df = pd.DataFrame(data)

df.head()

In [ ]:
df.info()

Como podemos ver usando el método df.info(), no tenemos ningún dato nulo en el dataframe.

Tenemos un data frame compuesto por 93597 filas y 11 columnas (work_year, experience_level, employment_type, job_title, salary, salary_currency, salary_in_usd, employee_residence, remote_ratio, company_location, company_size).

Para este proyecto vamos a eliminar las siguientes columnas ya que no nos son necesarias:
- salary (Ya que vamos a usar salary_in_usd para que todos los salarios estén ajustados a un mismo tipo de moneda)
- salary_currency (Como vamos a usar salary_in_usd esta columna es redundante)
- employee_residence (No necesitamos estos datos para los análisis que vamos a realizar)
- company_location (No necesitamos estos datos para los análisis que vamos a realizar)

In [ ]:
df_clean = df.drop(
    columns=['salary', 'salary_currency', 'employee_residence', 'company_location']
)
df_clean

### Distribución por tamaño de las empresas

In [ ]:
company_percentage = (
    df_clean['company_size'].value_counts(normalize=True).reindex(['S','M','L']) * 100
)

company_percentage

In [ ]:
fig_bar = go.Figure()

fig_bar.add_trace(go.Bar(
    x=company_percentage.index,
    y=company_percentage.values,
    marker_color=['#3B0F70','#8C2981','#DE4968'],
    texttemplate='%{y:.2f}%',
))

fig_bar.update_layout(
     title=dict(
        text='Distribución % por Tamaño de Empresa',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Tamaño de Empresa (Small/Medium/Large)',
        font=dict(size=20)
    ),
    yaxis_title=dict(
        text='Recuento %',
        font=dict(size=18)
    ),
)

### Conclusiones:
Dado que el tamaño de empresa es una variable categórica, se utilizó un gráfico de barras para analizar su distribución.

La mayoría de los registros corresponden a empresas de tamaño medio, representando casi el 97% del total. Las empresas grandes comprenden aproximadamente un 3%, mientras que las pequeñas constituyen la menor proporción del dataset, con apenas un 0,23%.

Esta distribución indica que los resultados del análisis pueden estar fuertemente influenciados por las empresas medianas, y que las conclusiones sobre empresas pequeñas o grandes deben interpretarse con cautela debido al reducido número de registros disponibles.

La alta proporción de empresas medianas podría deberse a que este tipo de empresas representan la mayoría de empleadores en el sector tecnológico o de datos, y suelen ser más activas en publicar ofertas de trabajo, lo que se refleja en los registros del dataset.

### Top 10 puestos de trabajo más comunes

In [ ]:
# Hacemos un recuento de cuantas veces aparece cada puesto de trabajo y escogemos los 10 primeros
job_counts = (
    df_clean['job_title']
    .value_counts()
    .head(10)
    .reset_index()
)

job_counts.columns = ['job_title', 'count']

# Gráfico
fig = go.Figure()
fig.add_trace(go.Bar(x=job_counts['count'],
                    y=job_counts['job_title'],
                    orientation='h',
                    marker=dict(
                       color=job_counts['count'],
                       colorscale='Plasma'
                    ),
                    text=job_counts['count'],
                    textposition='auto'
))

# Titulo y titulos de los ejes
fig.update_layout(
    title=dict(
        text='Top 10 puestos de trabajo más comunes',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Número de registros',
        font=dict(size=16)
    ),
    yaxis_title=dict(
        text='Puesto de trabajo',
        font=dict(size=16)
    ),
    yaxis=dict(autorange='reversed'),
    template='simple_white'
)

### Conclusiones:
El puesto más común es Data Scientist, seguido de Data Engineer, lo que indica una alta demanda en estas áreas, probablemente impulsada por la expansión del Big Data y la Inteligencia Artificial.

El tercer puesto más frecuente es Software Engineer, lo que refleja que, independientemente de las tendencias tecnológicas, siempre existe una necesidad significativa de profesionales dedicados al desarrollo y mantenimiento de software.

### ¿Como varía el salario según la experiencia?

In [ ]:
# Boxplot
# Averiguamos que tipo de niveles de experiencia tenemos en el dataset
pd.unique(df_clean['experience_level'])

# Convertimos la columna experience_level de object a category para que sea más eficiente en el uso futuro para las representaciones gráficas.
df_clean['experience_level'] = df_clean['experience_level'].astype("category").cat.set_categories(['EN', 'MI', 'SE', 'EX'], ordered=True)

In [ ]:
# Comprobamos de nuevo la columna
pd.unique(df_clean['experience_level'])

In [ ]:
# Representación gráfica mediante BoxPlot
boxfig = go.Figure(data=go.Box(x=df_clean['experience_level'],
                              y=df_clean['salary_in_usd'],
                              )
                  )

# Personalización de titulos y ejes
boxfig.update_layout(
    title=dict(
        text='Rangos Salariales según Nivel de Experiencia',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Nivel de Experiencia',
        font=dict(size=16)
    ),
    yaxis_title=dict(
        text='Rango Salarial en $',
        font=dict(size=16)
    ),
    xaxis=dict(categoryorder='array', 
               categoryarray=['EN','MI','SE','EX'],
               tickvals=['EN','MI','SE','EX'],
               ticktext=['Entry', 'Middle' ,'Senior' ,'Executive']),
    yaxis=dict(range=[0,500000])
)

### Conclusiones:
Se observa una evolución clara de los salarios en función del nivel de experiencia. Aunque algunos salarios de Seniors se acercan a los de los Ejecutivos, en general existen diferencias salariales notables entre cada nivel de experiencia.

Al pasar del nivel Junior al nivel Middle, la mediana salarial aumenta de aproximadamente \\$88.000 a \\$130.000, lo que representa un incremento significativo

### Media Salarial según tamaño de Empresa

In [ ]:
# Convertimos la columna company_size a una columna categórica para poder trabajar mejor con ella y la reordenamos
df_clean['company_size'] = df_clean['company_size'].astype("category").cat.set_categories(['S', 'M', 'L'], ordered=True)

# Recogemos el tamaño de las empresas y las medias salariales en una variable para tener un acceso más fácil y simple
company_mean_salary = df_clean.groupby('company_size')['salary_in_usd'].mean()
company_mean_salary

In [ ]:
# Realizamos el gráfico
fig3 = go.Figure()

# Barra empresas pequeñas
fig3.add_trace(go.Bar(
    x=['Pequeña'],
    y=[company_mean_salary['S']],
    width=0.6,
    name='Pequeña',
    text=[company_mean_salary['S']],
    texttemplate='$%{text:,.0f}',
    marker_color='#3B0F70'
))

# Barra empresas medianas
fig3.add_trace(go.Bar(
    x=['Mediana'],
    y=[company_mean_salary['M']],
    width=0.6,
    name='Mediana',
    text=[company_mean_salary['M']],
    texttemplate='$%{text:,.0f}',
    marker_color='#8C2981'
))

# Barra empresas grandes
fig3.add_trace(go.Bar(
    x=['Grande'],
    y=[company_mean_salary['L']],
    width=0.6,
    name='Grande',
    text=[company_mean_salary['L']],
    texttemplate='$%{text:,.0f}',
    marker_color='#DE4968'
))

# Personalización de titulos y ejes
fig3.update_layout(
    title=dict(
        text='Media Salarial Según Tamaño de Empresa',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Tamaño Empresa',
        font=dict(size=16)
    ),
    yaxis_title=dict(
        text='Media Salarial en $',
        font=dict(size=16)
    ),
    yaxis=dict(dtick=20000)
)

### Conclusiones:
Tanto las empresas medianas como las grandes tienden a ofrecer rangos salariales similares, con una ligera ventaja de las medianas, aunque no significativa.

Las empresas pequeñas se sitúan claramente por debajo, con salarios medios aproximadamente la mitad de los de medianas o grandes. Esto probablemente se deba a que, al ser empresas más reducidas o start-ups, cuentan con menos presupuesto y clientes más pequeños, lo que limita los recursos disponibles para remunerar a sus empleados.

### ¿Cómo evoluciona el salario desde 2020 hasta 2025?

In [ ]:
# Primero necesito crear una columna nueva para las modalidades de trabajo, en nuestro caso Presencial, Híbrido y Remoto.
# Usaremos pd.cut
bins = [-1,0,99,101]
labels = ['Presencial', 'Híbrido' , 'Remoto']

df_clean['work_mode'] = pd.cut(df_clean['remote_ratio'], bins=bins, labels=labels)

In [ ]:
# Ahora creamos un nuevo dataframe para poder manejar mejor los datos
df_evolution = df_clean.groupby(['work_year', 'work_mode'], observed=True)['salary_in_usd'].mean().unstack()

df_evolution = df_evolution.reset_index()

df_evolution

In [ ]:
# Creamos la gráfica
fig4 = go.Figure()

fig4.add_trace(go.Scatter(x=df_evolution['work_year'],
                        y=df_evolution['Presencial'],
                        mode='lines+markers',
                        name='Presencial'))

fig4.add_trace(go.Scatter(x=df_evolution['work_year'],
                        y=df_evolution['Híbrido'],
                        mode='lines+markers',
                        name='Híbrido'))
fig4.add_trace(go.Scatter(x=df_evolution['work_year'],
                        y=df_evolution['Remoto'],
                        mode='lines+markers',
                        name='Remoto'))

fig4.update_layout(
        title=dict(
        text='Evolución salarial según Modalidad de Empleo',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Año',
        font=dict(size=20)
    ),
    yaxis_title=dict(
        text='Media Salarial en $',
        font=dict(size=18)
    ),
    yaxis=dict(dtick=20000)
)



### Conclusiones:
La modalidad presencial suele ser la mejor remunerada, seguida de cerca por la modalidad remota. En general, todas las modalidades muestran un incremento de salarios entre 2020 y 2025, lo que refleja un crecimiento económico general.

La modalidad híbrida presenta valores más bajos, probablemente debido a un número reducido de registros en nuestros datos, lo que sugiere una menor oferta o demanda de puestos híbridos durante este período.

### ¿Cómo ha evolucionado el trabajo en remoto desde 2020 hasta 2025?

In [ ]:
# Preparamos los datos
df_remote_ratio = df_clean[df_clean['work_mode'] == 'Remoto'].groupby('work_year')['work_mode'].count()

df_total = df_clean.groupby('work_year')['work_mode'].count()

remote_percentage = (df_remote_ratio / df_total) * 100

In [ ]:
remote_percentage

In [ ]:
# Creamos la gráfica
fig5 = go.Figure()

fig5.add_trace(go.Scatter(x=remote_percentage.index,
                         y=remote_percentage.values,
                         mode='lines+markers',
                         name='Evolución'))

fig5.update_layout(
    title=dict(
        text='Evolución Del trabajo Remoto a lo largo del tiempo',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Año',
        font=dict(size=20)
    ),
    yaxis_title=dict(
        text='% de Trabajos Remoto',
        font=dict(size=18)
    ),
)


### Conclusiones:
A partir de 2022 se aprecia un descenso en el número de puestos con modalidad remota, coincidiendo con la finalización de la etapa más crítica del COVID. Entre 2022 y 2024 se observa un desplome notable, tras el cual la tendencia hacia los puestos remotos comienza a recuperarse, aunque de manera leve.

### Comparación salarios modalidad presencial vs modalidad remota

### Juniors presencial vs Juniors remoto

In [ ]:
# Filtramos los juniors de nuestro dataframe para poder trabajar más comodamente
df_juniors = df_clean[df_clean['experience_level']=='EN']

# Agrupamos por año y modalidad de trabajo
df_grouped = df_juniors.groupby(['work_year', 'work_mode'], observed=True)['salary_in_usd'].median().unstack()

df_grouped = df_grouped.reset_index()

df_grouped

In [ ]:
# Dibujamos la gráfica
fig6 = go.Figure()

fig6.add_trace(go.Scatter(
    x=df_grouped['work_year'],
    y=df_grouped['Presencial'],
    mode='lines+markers',
    name='Presencial')
)

fig6.add_trace(go.Scatter(
    x=df_grouped['work_year'],
    y=df_grouped['Remoto'],
    mode='lines+markers',
    name='Remoto')
)

fig6.update_layout(
    title=dict(
        text='Evolución salarial Junior Presencial vs Remoto',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Año',
        font=dict(size=20)
    ),
    yaxis_title=dict(
        text='Mediana Salarial en $',
        font=dict(size=18)
    ),
)

In [ ]:
df_juniors.groupby(['work_year','work_mode'], observed=True).size()

### Conclusiones:
En los primeros años, la remuneración presencial para juniors era superior a la modalidad remota. A lo largo del tiempo, los salarios se han ido normalizando, y en el último año la modalidad remota incluso supera ligeramente a la presencial.

Entre 2020 y 2021 se observa un salto importante en la modalidad presencial, debido al bajo número de registros en esos años. Posteriormente, con un mayor número de registros, los datos se estabilizan y la evolución salarial refleja mejor la tendencia real.

### Seniors presencial vs Seniors remoto

In [ ]:
# Filtramos los seniors de nuestro dataframe para poder trabajar más comodamente
df_seniors = df_clean[df_clean['experience_level']=='SE']

# Agrupamos por año y modalidad de trabajo
df_grouped_se = df_seniors.groupby(['work_year', 'work_mode'], observed=True)['salary_in_usd'].median().unstack()

df_grouped_se = df_grouped_se.reset_index()

df_grouped_se

In [ ]:
# Dibujamos la gráfica
fig7 = go.Figure()

fig7.add_trace(go.Scatter(
    x=df_grouped_se['work_year'],
    y=df_grouped_se['Presencial'],
    mode='lines+markers',
    name='Presencial')
)

fig7.add_trace(go.Scatter(
    x=df_grouped_se['work_year'],
    y=df_grouped_se['Remoto'],
    mode='lines+markers',
    name='Remoto')
)

fig7.update_layout(
    title=dict(
        text='Evolución salarial Senior Presencial vs Remoto',
        y=0.85,
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(size=25)
    ),
    xaxis_title=dict(
        text='Año',
        font=dict(size=20)
    ),
    yaxis_title=dict(
        text='Mediana Salarial en $',
        font=dict(size=18)
    ),
)

In [ ]:
df_seniors.groupby(['work_year','work_mode'], observed=True).size()

### Conclusiones:
Como podemos observar, en los puestos Seniors la modalidad presencial suele estar ligeramente mejor remunerada que la modalidad remota. Se aprecia un salto importante en los salarios presenciales entre 2020 y 2021, lo cual se debe a que en 2020 el número de registros era muy reducido, provocando que la mediana inicial fuera baja. En los años posteriores, con un número de registros más elevado, los datos se estabilizan y la evolución salarial se vuelve más representativa.

### Conclusiones finales

En general, los análisis muestran que los salarios en el sector de datos y tecnología crecen de forma consistente con la experiencia y el tiempo. La modalidad presencial tiende a estar ligeramente mejor remunerada que la remota, aunque esta última ha ganado importancia en los últimos años.

Los puestos más demandados siguen siendo Data Scientist y Data Engineer, seguidos de Software Engineer, reflejando la alta necesidad de profesionales en Big Data, Inteligencia Artificial y desarrollo de software.

Las empresas medianas dominan el dataset, lo que indica que son los principales empleadores en este sector, mientras que las pequeñas y grandes están menos representadas. Esta distribución sugiere que los resultados pueden estar influenciados por la predominancia de las medianas y que los hallazgos sobre pequeñas y grandes empresas deben interpretarse con cautela.

Por último, el análisis muestra que los salarios se correlacionan positivamente con la experiencia y el nivel de responsabilidad, y que la modalidad de trabajo y el tamaño de la empresa son factores relevantes que impactan en la remuneración. Sin embargo, es importante tener en cuenta las limitaciones del dataset, como el reducido número de registros en algunos años o modalidades, que pueden afectar la representatividad de ciertos hallazgos.